# 15 高斯混合模型 Gaussian Mixture Model

依赖安装说明：`pip install numpy matplotlib scikit-learn`

GMM 假设数据来自多个高斯分布的混合。相比 K-Means 的硬分配，GMM 会给出“每个点属于每个簇的概率”。


## 1. 数学逻辑

混合模型的概率密度：

$$p(x)=\sum_{k=1}^{K}\pi_k \mathcal{N}(x|\mu_k,\Sigma_k)$$

其中 `pi_k` 是第 k 个高斯成分的权重。

EM 算法两步循环：

E step：计算责任度 responsibility：

$$r_{ik}=\frac{\pi_k\mathcal{N}(x_i|\mu_k,\Sigma_k)}{\sum_j\pi_j\mathcal{N}(x_i|\mu_j,\Sigma_j)}$$

M step：用责任度加权更新参数。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

np.random.seed(42)
X, _ = make_blobs(n_samples=260, centers=[[-2, 0], [1.5, 1.0], [2, -1.5]], cluster_std=[0.5, 0.8, 0.35], random_state=42)


In [ ]:
# 从零实现：一维 GMM 的 EM，方便看清责任度
x = np.r_[np.random.normal(-2, 0.5, 100), np.random.normal(2, 0.8, 120)]
K = 2
pi = np.ones(K) / K
mu = np.array([-1.0, 1.0])
sigma = np.array([1.0, 1.0])

def normal_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

for step in range(20):
    resp = np.column_stack([pi[k] * normal_pdf(x, mu[k], sigma[k]) for k in range(K)])
    resp = resp / resp.sum(axis=1, keepdims=True)
    Nk = resp.sum(axis=0)
    pi = Nk / len(x)
    mu = (resp * x[:, None]).sum(axis=0) / Nk
    sigma = np.sqrt((resp * (x[:, None] - mu) ** 2).sum(axis=0) / Nk)

print('pi:', np.round(pi, 3))
print('mu:', np.round(mu, 3))
print('sigma:', np.round(sigma, 3))


In [ ]:
model = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
labels = model.fit_predict(X)
proba = model.predict_proba(X)
print('每个成分权重:', np.round(model.weights_, 3))
print('前 3 个点的簇概率:')
print(np.round(proba[:3], 3))

plt.scatter(X[:,0], X[:,1], c=labels, cmap='tab10', s=24)
plt.scatter(model.means_[:,0], model.means_[:,1], c='red', marker='x', s=120)
plt.title('GaussianMixture 软聚类')
plt.show()


## 2. 常见误区

- GMM 假设每个簇近似高斯形状，不适合任意形状簇。
- EM 可能陷入局部最优，初始化会影响结果。
- 协方差类型越灵活，模型越容易过拟合。

## 3. 小实验

- 改 `n_components`。
- 改 `covariance_type` 为 `diag` 或 `spherical`。
- 观察边界附近样本的概率分布。
